# Analisis Pendapatan POS 2024 vs 2025
Dataset: `paket_a_transaksi_pos_2024_2025.csv`

**Aturan pendapatan:** `pendapatan = jumlah × harga_satuan × (1 − diskon_persen/100)`, hanya untuk transaksi berstatus **SELESAI**.
`BATAL` dan `RETUR` tidak dihitung sebagai pendapatan.

In [1]:
import pandas as pd
pd.options.display.float_format = '{:,.0f}'.format
df = pd.read_csv('paket_a_transaksi_pos_2024_2025.csv', dtype=str)
print(df.shape)
df.head()

(5030, 9)


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,"7500,00",0,SELESAI
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500,0,SELESAI
4,TRX0243509,26/03/2025,G05,K016,RMT1407,1,6000,0,SELESAI


## 1. Pengecekan awal kualitas data

In [2]:
print('Missing value:'); print(df.isna().sum())
print('\nBaris duplikat persis:', df.duplicated().sum())
print('ID transaksi duplikat:', df.id_transaksi.duplicated().sum())
print('\nFormat tanggal:'); print(df.tanggal.str.replace(r'\d','9',regex=True).value_counts())
print('\nFormat harga:'); print(df.harga_satuan.str.replace(r'\d','9',regex=True).value_counts())
print('\nStatus:'); print(df.status.value_counts())

Missing value:
id_transaksi       0
tanggal            0
kode_gerai         0
kode_kasir       261
kode_produk        0
jumlah             0
harga_satuan       0
diskon_persen      0
status             0
dtype: int64

Baris duplikat persis: 30
ID transaksi duplikat: 30

Format tanggal:
tanggal
9999-99-99    4839
99-99-9999     105
99/99/9999      86
Name: count, dtype: int64

Format harga:
harga_satuan
99999       3974
9999         960
99999,99      40
99.999        38
9999,99       11
9.999          7
Name: count, dtype: int64

Status:
status
SELESAI    4749
BATAL       191
RETUR        90
Name: count, dtype: int64


## 2. Pembersihan data
- **Duplikat persis** → dihapus (agar tidak dihitung dua kali).
- **Tanggal** campuran `YYYY-MM-DD`, `DD/MM/YYYY`, `DD-MM-YYYY` → diseragamkan (day-first untuk format non-ISO).
- **Harga** campuran `11000,00` (koma desimal) dan `28.000` (titik ribuan) → diubah ke angka.
- **kode_kasir kosong** → diisi `TIDAK DIKETAHUI` (tidak memengaruhi pendapatan).
- **Status** → dirapikan (trim + huruf besar).

In [3]:
d = df.drop_duplicates().copy()

# tanggal
iso = d.tanggal.str.match(r'^\d{4}-\d{2}-\d{2}$')
d['tgl'] = pd.NaT
d['tgl'] = d['tgl'].astype('datetime64[ns]')
d.loc[iso, 'tgl'] = pd.to_datetime(d.loc[iso, 'tanggal'], format='%Y-%m-%d')
d.loc[~iso, 'tgl'] = pd.to_datetime(d.loc[~iso, 'tanggal'].str.replace('/', '-'), format='%d-%m-%Y')
d['tahun'] = d.tgl.dt.year

# harga
def parse_harga(s):
    s = s.strip()
    if ',' in s:                      # '11000,00' -> 11000.00
        return float(s.replace('.', '').replace(',', '.'))
    if '.' in s:                      # '28.000' -> 28000
        return float(s.replace('.', ''))
    return float(s)
d['harga'] = d.harga_satuan.map(parse_harga)

d['jumlah'] = d.jumlah.astype(int)
d['diskon_persen'] = d.diskon_persen.astype(int)
d['status'] = d.status.str.strip().str.upper()
d['kode_kasir'] = d.kode_kasir.fillna('TIDAK DIKETAHUI')

d['pendapatan'] = d.jumlah * d.harga * (1 - d.diskon_persen/100)
print('Setelah hapus duplikat:', len(d), 'baris | missing tersisa:', d.isna().sum().sum())
print(d.tahun.value_counts().sort_index())

Setelah hapus duplikat: 5000 baris | missing tersisa: 0
tahun
2024    2492
2025    2508
Name: count, dtype: int64


## 3. Anomali yang perlu diwaspadai
- **jumlah negatif** pada status SELESAI → kemungkinan koreksi/retur yang salah status; tidak dihitung sebagai penjualan.
- **jumlah sangat besar** (> 12, padahal pola normal maksimal 12) → hampir pasti salah input; dikeluarkan dari perhitungan utama dan dilaporkan.

In [4]:
print('Jumlah per nilai (SELESAI):'); print(d[d.status=='SELESAI'].jumlah.value_counts().sort_index().to_dict())
neg = d[(d.status=='SELESAI') & (d.jumlah<0)]
outl = d[(d.status=='SELESAI') & (d.jumlah>12)]
print('\nSELESAI jumlah negatif:', len(neg), '| nilai:', f'{neg.pendapatan.sum():,.0f}')
print('SELESAI jumlah >12 (outlier):', len(outl), '| nilai:', f'{outl.pendapatan.sum():,.0f}')
outl[['id_transaksi','tanggal','kode_produk','jumlah','harga','diskon_persen','pendapatan']]

Jumlah per nilai (SELESAI):
{-12: 1, -6: 2, -4: 3, -3: 4, -2: 12, -1: 14, 1: 1847, 2: 1139, 3: 692, 4: 506, 6: 344, 12: 144, 641: 1, 746: 1, 759: 1, 892: 1, 1280: 1, 1301: 1, 1310: 1, 1456: 1, 1983: 1, 2093: 1, 2375: 1}

SELESAI jumlah negatif: 36 | nilai: -2,108,325
SELESAI jumlah >12 (outlier): 11 | nilai: 383,023,225


,id_transaksi,tanggal,kode_produk,jumlah,harga,diskon_persen,pendapatan
87,TRX0243886,2025-12-27,PRW1306,2375,"14,500",0,"34,437,500"
244,TRX0243257,2024-11-24,MKN1102,1456,"32,500",20,"37,856,000"
467,TRX0243696,2024-02-27,SGR1201,892,"54,000",0,"48,168,000"
926,TRX0240632,2024-09-08,RMT1403,1280,"6,500",0,"8,320,000"
994,TRX0241002,2024-03-18,MKN1101,2093,"48,000",10,"90,417,600"
1035,TRX0241068,2025-09-17,SGR1202,759,"13,500",5,"9,734,175"
1323,TRX0240401,2025-02-02,MIN1009,641,"11,500",0,"7,371,500"
2414,TRX0241473,2024-12-08,PRW1308,1310,"11,000",0,"14,410,000"
3220,TRX0242632,2025-11-19,SGR1204,746,"41,500",0,"30,959,000"
3393,TRX0241029,2025-05-12,SGR1205,1983,"23,000",10,"41,048,100"


## 4. Pendapatan per tahun

In [5]:
valid = d[(d.status=='SELESAI') & (d.jumlah>0) & (d.jumlah<=12)]
hasil = (valid.groupby('tahun')
         .agg(jumlah_transaksi=('id_transaksi','count'), pendapatan=('pendapatan','sum')))
hasil.loc['TOTAL'] = hasil.sum()
hasil

,jumlah_transaksi,pendapatan
tahun,,
2024,"2,327","127,406,475"
2025,"2,345","135,001,100"
TOTAL,"4,672","262,407,575"


### Pengaruh keputusan pembersihan terhadap hasil (sensitivitas)

In [6]:
def per_tahun(x): return x.groupby('tahun').pendapatan.sum()
sens = pd.DataFrame({
  '1. Utama (SELESAI, jumlah 1-12)': per_tahun(valid),
  '2. Semua status (salah: abaikan status)': per_tahun(d),
  '3. SELESAI saja, tanpa buang outlier': per_tahun(d[d.status=='SELESAI']),
  '4. Utama + SELESAI bernilai negatif': per_tahun(d[(d.status=='SELESAI') & (d.jumlah<=12)]),
  '5. Utama tanpa diskon (salah: abaikan diskon)': (valid.jumlah*valid.harga).groupby(valid.tahun).sum(),
})
sens.T

tahun,2024,2025
"1. Utama (SELESAI, jumlah 1-12)","127,406,475","135,001,100"
2. Semua status (salah: abaikan status),"393,226,200","271,871,425"
"3. SELESAI saja, tanpa buang outlier","386,237,375","257,085,100"
4. Utama + SELESAI bernilai negatif,"126,764,425","133,534,825"
5. Utama tanpa diskon (salah: abaikan diskon),"130,883,500","138,664,000"


In [7]:
print('Pendapatan per status dan tahun:')
d.pivot_table(index='status', columns='tahun', values='pendapatan', aggfunc='sum')

Pendapatan per status dan tahun:


tahun,2024,2025
status,,
BATAL,"4,847,850","11,953,700"
RETUR,"2,140,975","2,832,625"
SELESAI,"386,237,375","257,085,100"


## 5. Simpan hasil

In [8]:
hasil.to_csv('hasil_pendapatan_per_tahun.csv')
d.drop(columns=['tgl']).to_csv('data_bersih.csv', index=False)
print('Tersimpan: hasil_pendapatan_per_tahun.csv, data_bersih.csv')

Tersimpan: hasil_pendapatan_per_tahun.csv, data_bersih.csv
